In [ ]:

# @title Gemini Medical Lecture Transcript → Structured Markdown Notes
# @markdown Enter a Google Drive recording link, Gemini API key, lecture name, model, and thinking level for each step, then run this one cell.
# @markdown The cell downloads the recording, runs STEP 1 verbatim transcription, runs STEP 2 Markdown structuring, and creates downloadable `.md` files.

!pip -q install -U google-genai gdown ipywidgets

import os
import re
import time
import mimetypes
from pathlib import Path
from IPython.display import display, Markdown, FileLink
import ipywidgets as widgets
import gdown
from google import genai
from google.genai import types

# Current Flash / Flash-Lite models shown in Google AI Studio / Gemini API docs as of 2026-07-28.
# Each model's available thinking levels are intentionally model-specific.
MODEL_THINKING_LEVELS = {
    "gemini-3.6-flash": ["minimal", "low", "medium", "high"],
    "gemini-3.5-flash": ["minimal", "low", "medium", "high"],
    "gemini-3.5-flash-lite": ["minimal", "low", "medium", "high"],
    "gemini-3.1-flash-lite": ["minimal", "low", "medium", "high"],
    "gemini-3-flash-preview": ["minimal", "low", "medium", "high"],
    "gemini-2.5-flash": ["low", "medium", "high"],
    "gemini-2.5-flash-lite": ["low", "medium", "high"],
}

FLASH_MODELS = list(MODEL_THINKING_LEVELS.keys())

STEP_1_PROMPT = """فرغ التسجيل ده من غير متوقع و لا كلمه منه. التسجيل عباره عن محاضره
بالعاميه المضريه و الإنجليزي و لازم تكتبها متشلش و لا كلمه و لا حديث و
لا نكته و لا غيره اكتب كل اللي هتسمعه عاوز تسجيل حرفي زي المدكتور اتكلم كل كلمه
بيقولها الدكتور مهمه

اللي العاميه المصريه اكتبه بيها و اللي بالأتجليزي اكتبه بالأنجليزي و هكذذا

متلخصش و لا تخصتصرو لا تكتب اهم النقط

اكتب كل حاجه

مترجمش عايمه مصريه لانجليزي ولا عاميه مصريه لعربي فصحى و لا انجليزي لعربي اكتب
كل حاجه زي متقالت

### Extra output rule for STEP 1

* Return the transcript in **Markdown format**.
* Make it **ready to download / save** as an `.md` document.
* Do not add any explanation or preamble.
* Start directly with the transcript."""

STEP_2_PROMPT_TEMPLATE = """[ROLE]

You are an expert medical transcript editor and formatter with deep
understanding of Egyptian Arabic and English medical terminology.

You will receive a complete verbatim transcript of a medical lecture. The
transcript may contain Egyptian Arabic, English medical terms, questions,
examples, jokes, repetitions, corrections, and spontaneous comments.

Your task is to organize and format the transcript into clear, study-ready
structured notes without removing, summarizing, paraphrasing, translating,
correcting, or adding any content.

[INPUT]

Lecture title: "{lecture_name}"

Transcript:

{transcript}

[PRIMARY OBJECTIVE]

Convert the provided verbatim transcript into well-structured Markdown notes
while preserving every piece of information and maintaining the exact
chronological order of the lecture.

The input transcript is the source of truth.

Do not use outside medical knowledge.

Do not infer missing information.

Do not rewrite the doctor's meaning.

[ZERO-LOSS RULE — ABSOLUTE]

You must preserve every sentence, explanation, example, clarification,
comparison, joke, anecdote, correction, side comment, question, answer, and
deliberate repetition contained in the transcript.

Do not:

* summarize;
* shorten;
* compress;
* paraphrase;
* simplify;
* translate;
* replace the doctor's wording with more professional wording;
* remove comments because they seem unimportant;
* remove repeated explanations when they add emphasis or clarification;
* add information that is not present in the transcript.

Every meaningful part of the input must appear in the output.

[LANGUAGE PRESERVATION]

* Keep Egyptian Arabic exactly in Egyptian Arabic.
* Keep English medical terminology in English.
* Keep the natural Egyptian Arabic-English mixed teaching style.
* Do not translate Egyptian Arabic into formal Arabic.
* Do not translate Egyptian Arabic into English.
* Do not translate English medical terminology into Arabic.
* Do not convert the doctor's colloquial expressions into formal language.
* Do not correct the doctor's grammar or speaking style unless the transcript
  contains an obvious transcription typo that can be corrected without
  changing the spoken wording.

[CHRONOLOGICAL ORDER]

Follow the exact sequence of the transcript.

Do not:

* move information to an earlier or later section;
* combine information from distant parts of the lecture;
* reorganize the lecture according to textbook order;
* place all symptoms, investigations, or treatments together if the doctor
  discussed them at different times.

The output must follow the doctor's original teaching flow.

[HEADINGS]

Add Markdown headings only at natural topic transitions.

Use:

Main Topic

Subtopic

Smaller Subtopic

Headings should be descriptive and based only on the content stated by the
doctor.

Do not create headings that introduce information not present in the transcript.

Do not place a heading after every sentence.

Use granular subheadings when the topic clearly changes, but preserve the
natural continuity of the lecture.

[STRUCTURAL ORGANIZATION]

Organize the transcript into paragraphs, bullet points, numbered lists, and
sub-bullets according to the doctor's actual speech.

1. Bullet points

Use bullet points when the doctor presents a group of separate items, such as:

* causes;
* symptoms;
* signs;
* risk factors;
* types;
* classifications;
* differential diagnoses;
* investigations;
* treatments;
* complications;
* advantages and disadvantages;
* comparisons;
* indications;
* contraindications.

Each distinct item should have its own bullet.

Place the doctor's explanation of an item beneath it as an indented sub-bullet.

Example structure:

* First item

  * The doctor's complete explanation of this item.
  * Any examples, clarifications, or additional comments related to it.

Do not shorten the wording to create concise bullet points.

2. Numbered lists

Use numbered lists only when order matters, such as:

* procedure steps;
* examination steps;
* chronological stages;
* disease progression;
* mechanisms occurring in sequence;
* priority order;
* management algorithms.

Detect ordered sequences from expressions such as:

* أول حاجة
* تاني حاجة
* بعد كده
* الخطوة اللي بعدها
* first
* second
* then
* next
* finally

Do not number unordered groups.

3. Narrative introductions followed by lists

When the doctor introduces a list using a flowing sentence:

* preserve the full introductory sentence;
* place the items beneath it as bullets or numbered steps;
* preserve all connective reasoning and transitional wording.

Do not remove words such as:

* لأن
* عشان
* يعني
* فبالتالي
* so
* because
* therefore

4. Narrative paragraphs

Keep continuous explanations as paragraphs when they are not genuine lists.

This includes:

* clinical reasoning;
* mechanisms;
* analogies;
* stories;
* jokes;
* exam advice;
* case discussions;
* clarifications;
* comparisons explained as continuous speech;
* spontaneous teaching comments.

Do not force every sentence into bullet points.

5. Sub-bullets

Use indented sub-bullets for:

* explanations of a main item;
* examples;
* mechanisms;
* consequences;
* exceptions;
* clarifications;
* additional details related to the main bullet.

Do not separate an explanation from the point it explains.

[REPETITIONS AND SPEECH CHARACTERISTICS]

Preserve deliberate repetitions when the doctor repeats an idea for:

* emphasis;
* clarification;
* correction;
* teaching reinforcement;
* comparison.

Do not delete complete repeated ideas merely because they appear similar.

You may remove only obvious accidental duplicated text caused by transcription
processing, such as the exact same sentence being pasted twice consecutively
when it clearly represents a technical duplication rather than spoken
repetition.

When uncertain, preserve it.

[SPEAKER LABELS]

Identify interactions when they are clear from the transcript.

Use:

[Student Question:]

[Doctor's Response:]

Preserve the complete wording of both the question and response.

Ignore nothing from an acknowledged student interaction.

Do not label ordinary sentences as student speech unless the transcript clearly
indicates a speaker change.

[UNCLEAR OR INCOMPLETE TEXT]

Preserve markers such as:

[unclear]

Do not guess or replace unclear words using medical knowledge.

If a sentence is incomplete in the transcript, keep it incomplete.

Do not complete unfinished statements from your own knowledge.

[MEDICAL ACCURACY]

Preserve medical terms, abbreviations, dosages, measurements, classifications,
drug names, anatomical terms, investigation names, and numerical values exactly
as written in the transcript.

Do not medically correct the doctor.

Do not silently replace a term because another term seems more accurate.

The task is transcript organization, not medical fact-checking.

[MARKDOWN FORMAT]

Use clean Markdown formatting:

* # for major headings;
* ## for subheadings;
* ### only when necessary;
* * for unordered items;
* 1. for genuinely ordered steps;
* two-space indentation for sub-bullets;
* bold text sparingly for terms that the doctor clearly introduces as central
  concepts.

Do not use tables unless the doctor explicitly presents information in a true
comparison that can be transferred into a table without changing, shortening, or
separating the wording.

Prefer bullets and paragraphs over tables because tables may force compression.

[FINAL VALIDATION — MANDATORY]

Before producing the final answer, internally verify that:

1. Every meaningful sentence from the input appears in the output.
2. No explanation, joke, example, question, answer, correction, or side comment
   was removed.
3. No content was summarized or paraphrased.
4. No medical information was added.
5. The original chronological order was preserved.
6. Egyptian Arabic and English terminology were preserved in their original
   languages.
7. Bullets and numbered lists reflect the doctor's real structure.
8. Narrative sections were not unnecessarily converted into lists.
9. The output remains detailed and complete, even if it becomes very long.

[OUTPUT]

Begin directly with the structured lecture notes.

Do not include:

* a preamble;
* an explanation of your work;
* a summary;
* a conclusion added by you;
* a list of changes;
* comments about formatting;
* statements such as “Here is the structured transcript.”

### Extra output rule for STEP 2

* Return the notes in **Markdown format**.
* Make it **ready to download / save** as an `.md` document.
* Start directly with the structured notes.
* Do not add any extra explanation."""

def safe_filename(name):
    cleaned = re.sub(r"[^\w\-.\u0600-\u06FF]+", "_", name.strip(), flags=re.UNICODE).strip("_")
    return cleaned or "lecture"

def download_drive_file(url):
    path = gdown.download(url=url, output=None, quiet=False, fuzzy=True)
    if not path:
        raise RuntimeError("Could not download the Google Drive file. Make sure the link is shared so anyone with the link can view it.")
    guessed = mimetypes.guess_extension(mimetypes.guess_type(path)[0] or "")
    return Path(path)

def wait_for_file_active(client, uploaded_file):
    name = uploaded_file.name
    while True:
        current = client.files.get(name=name)
        state = getattr(current, "state", None)
        state_name = getattr(state, "name", str(state))
        if "ACTIVE" in state_name:
            return current
        if "FAILED" in state_name:
            raise RuntimeError(f"Gemini file processing failed: {state_name}")
        time.sleep(5)

def make_config(thinking_level):
    # Supports the current thinking-level API; fallbacks below handle older SDK surface differences.
    return types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_level=thinking_level),
        response_mime_type="text/plain",
    )

def generate_text(client, model, thinking_level, contents):
    try:
        response = client.models.generate_content(
            model=model,
            contents=contents,
            config=make_config(thinking_level),
        )
    except Exception as first_error:
        try:
            response = client.models.generate_content(
                model=model,
                contents=contents,
                config={"thinking_config": {"thinking_level": thinking_level}, "response_mime_type": "text/plain"},
            )
        except Exception:
            raise first_error
    text = getattr(response, "text", None) or ""
    if not text.strip():
        raise RuntimeError("Gemini returned an empty response.")
    return text

# Inputs
api_key = widgets.Password(description="Gemini API key:", layout=widgets.Layout(width="700px"))
drive_link = widgets.Text(description="Drive link:", placeholder="https://drive.google.com/file/d/.../view", layout=widgets.Layout(width="900px"))
lecture_name = widgets.Text(description="Lecture name:", value="medical_lecture", layout=widgets.Layout(width="700px"))

step1_model = widgets.Dropdown(description="Step 1 model:", options=FLASH_MODELS, value="gemini-3.6-flash")
step1_thinking = widgets.Dropdown(description="Step 1 thinking:", options=MODEL_THINKING_LEVELS[step1_model.value], value="minimal")
step2_model = widgets.Dropdown(description="Step 2 model:", options=FLASH_MODELS, value="gemini-3.6-flash")
step2_thinking = widgets.Dropdown(description="Step 2 thinking:", options=MODEL_THINKING_LEVELS[step2_model.value], value="low")
run_button = widgets.Button(description="Run 2-step workflow", button_style="success")
output = widgets.Output()

def sync_thinking_options(model_widget, thinking_widget, preferred):
    def _update(change):
        levels = MODEL_THINKING_LEVELS[change["new"]]
        thinking_widget.options = levels
        thinking_widget.value = preferred if preferred in levels else levels[0]
    model_widget.observe(_update, names="value")

sync_thinking_options(step1_model, step1_thinking, "minimal")
sync_thinking_options(step2_model, step2_thinking, "low")

def run_workflow(_):
    with output:
        output.clear_output()
        if not api_key.value.strip() or not drive_link.value.strip():
            raise ValueError("Please enter both your Gemini API key and Google Drive link.")
        os.environ["GEMINI_API_KEY"] = api_key.value.strip()
        client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
        base = safe_filename(lecture_name.value)

        print("Downloading Google Drive recording...")
        local_file = download_drive_file(drive_link.value.strip())
        print(f"Downloaded: {local_file}")

        print("Uploading recording to Gemini File API...")
        uploaded = client.files.upload(file=str(local_file))
        uploaded = wait_for_file_active(client, uploaded)

        print(f"STEP 1: Transcribing with {step1_model.value} / thinking={step1_thinking.value}...")
        transcript = generate_text(client, step1_model.value, step1_thinking.value, [uploaded, STEP_1_PROMPT])
        transcript_path = Path(f"{base}_verbatim_transcript.md")
        transcript_path.write_text(transcript, encoding="utf-8")
        print(f"Saved transcript: {transcript_path}")

        print(f"STEP 2: Structuring notes with {step2_model.value} / thinking={step2_thinking.value}...")
        step2_prompt = STEP_2_PROMPT_TEMPLATE.format(lecture_name=lecture_name.value.strip() or base, transcript=transcript)
        notes = generate_text(client, step2_model.value, step2_thinking.value, step2_prompt)
        notes_path = Path(f"{base}_structured_notes.md")
        notes_path.write_text(notes, encoding="utf-8")
        print(f"Saved notes: {notes_path}")

        display(Markdown("## Download files"))
        display(FileLink(str(transcript_path)))
        display(FileLink(str(notes_path)))

run_button.on_click(run_workflow)

display(Markdown("# Gemini Medical Lecture: recording → transcript → structured Markdown notes"))
display(Markdown("Models are limited to current Flash and Flash-Lite options, with thinking choices updated per selected model."))
display(api_key, drive_link, lecture_name, step1_model, step1_thinking, step2_model, step2_thinking, run_button, output)
